#### Tecnologie dei dati e del linguaggio
# Dati e informazione
### Prof. Alfio Ferrara


## Introduzione al trattamento del dato testuale
I dati sono tratti dal [dataset online](https://www.kaggle.com/datasets/edoardoscarpaci/italian-food-recipes)

In [ ]:
import pandas as pd
import numpy as np
from data.recipedata import ItalianRecipes

In [ ]:
R = ItalianRecipes(file_path="/Users/Flint/Data/recipes/it_recipes.csv")

In [ ]:
docs = [x for x in R.df['Steps'].values if not pd.isnull(x)]

In [ ]:
for doc in docs[:10]:
    print(f"{doc}\n")

**Che tipo di operazioni e applicazioni vogliamo realizzare coi dati testuali?**


Per capire quanto una operazione può essere difficile per una macchina, proviamo a sforzarci di "vedere" il testo come lo vede la macchina

In [ ]:
from nlp.utils import messy_text

In [ ]:
for text in docs[:10]:
    print(f"{messy_text(text)}\n")

## Vector Space Model
Un modo di trasformare il testo in qualcosa che la macchina può computare possiamo individuare una forma vettoriale per il testo usando alcune sue caratteristiche.

In [ ]:
from nltk.tokenize import sent_tokenize

In [ ]:
example = "\n".join(sent_tokenize(docs[0]))
print(example)

In [ ]:
print(example.split())

In [ ]:
def features(text):
    return np.array([len(text), len([x for x in text if x.isupper()]) / len(text)])

In [ ]:
D = np.array([features(d) for d in docs])
D 

In [ ]:
import matplotlib.pyplot as plt

**Scaliamo i dati**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler()
S = scaler.fit_transform(D)
row_norms = np.linalg.norm(S, axis=1, keepdims=True)
row_norms[row_norms == 0] = 1
U = S / row_norms

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), ncols=3)
ax[0].scatter(D[:,0], D[:,1], s=4, alpha=.4)
ax[0].set_ylim((D.min(), D.max()+1))
ax[0].set_title('Raw data')
ax[1].scatter(S[:,0], S[:,1], s=4, alpha=.4)
ax[1].set_title('Scaled data (MinMax)')
ax[2].scatter(U[:,0], U[:,1], s=4, alpha=.4)
ax[2].set_title('Unitary vectors')
plt.tight_layout()
plt.show()

## Usare le parole come dimensioni

### Tokenizzazione per espressioni regolari

In [ ]:
from nltk.tokenize import word_tokenize

In [ ]:
for doc in docs[:10]:
    print(word_tokenize(doc, language='italian'))

### Tokenizzatori basati su modelli linguistici

In [ ]:
import spacy
from spacy.displacy import render

In [ ]:
nlp = spacy.load('it_core_news_lg')

In [ ]:
tokens = []
for token in nlp(docs[0]):
    tokens.append({'text': token.text, 'lemma': token.lemma_, 'pos': token.pos_, 
                   'tag': token.tag_, 'dep': token.dep_, 'shape': token.shape_, 
                   'alpha': token.is_alpha, 'stopword': token.is_stop})
T = pd.DataFrame(tokens)

In [ ]:
T.head(20)

In [ ]:
sentence = "Per preparare il tiramisù preparate il caffé con la moka"
render(nlp(sentence))

In [ ]:
def spacy_tokenizer(text, lowercase=True, lemma=True, keep_pos=None):
    def selector(token, lemma=lemma, keep_pos=keep_pos):
        if keep_pos is None or token.pos_ in keep_pos:
            if lemma:
                return token.lemma_ 
            else:
                return token.text 
    tokens = []
    for token in nlp(text):
        s = selector(token)
        if s is not None:
            if lowercase:
                tokens.append(s.lower())
            else:
                tokens.append(s)
    return tokens

In [ ]:
for doc in docs[:10]:
    print(spacy_tokenizer(doc, lowercase=True, lemma=True, keep_pos=['PROPN', 'VERB', 'ADJ', 'ADV', 'AUX']))

### Size of the vocabulary with different types of tokenizers

In [ ]:
from collections import defaultdict
from tqdm.notebook import tqdm

In [ ]:
nltk_vocabulary = defaultdict(lambda: 0)
spacy_vocabulary = defaultdict(lambda: 0)
step = 10

def counter(docs: list, tokenizer: callable, vocabulary: dict, step: int = 10):
    run = list(enumerate(docs))
    stats = []
    for i, doc in tqdm(run):
        if i % step == 0:
            stats.append(len(vocabulary))
        for token in tokenizer(doc):
            vocabulary[token] += 1
    return stats 

nltk_stats = counter(docs, word_tokenize, nltk_vocabulary, step=step)
spacy_stats = counter(docs, spacy_tokenizer, spacy_vocabulary, step=step)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.set_title('Vocabulary Size')
ax.plot(nltk_stats, label='NLTK')
ax.plot(spacy_stats, label='spaCy')
ax.legend()
plt.tight_layout()
plt.show()

### Un esempio di motore di ricerca
Utilizziamo la tokenizzazione per costruire un semplice motore di ricerca:
- tokenizzatore
- vettorizzatore
- misuratore di distanze

In [35]:
from sklearn.feature_extraction.text import CountVectorizer

In [36]:
def tokenize(text):
    return [x.lower() for x in word_tokenize(text, language='italian')]

In [38]:
vectorizer = CountVectorizer(tokenizer=tokenize, token_pattern=None)
X_sparse = vectorizer.fit_transform(docs)
X = X_sparse.toarray()

vocabulary = vectorizer.get_feature_names_out()
Xdf = pd.DataFrame(X, columns=vocabulary)

In [44]:
Xdf.head()

,!,%,&,','','mpacchiuse,'ncasciata,'nduja,'s,(,...,éclair,ìo,–,–tenendone,‘,’,“,”,…,…buon
0,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,2,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,2,...,0,0,0,0,0,2,0,0,0,0
3,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,3,0,0,0,0


In [46]:
#Xdf.loc[10].sort_values(ascending=False).head(20)

In [40]:
X.shape 

(5935, 20106)

In [43]:
print(X) 

[[1 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [1 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [47]:
q = "un dolce con mascarpone, uova e zucchero e vaniglia con anche la meringa"
qv = vectorizer.transform([q]).toarray()
#pd.Series(qv[0], index=vocabulary).sort_values(ascending=False)

In [48]:
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

In [51]:
search = cosine_similarity(qv, X)
sigma = pd.Series(search[0]).sort_values(ascending=False)
sigma.head(6)

4577    0.580695
2131    0.562059
457     0.558849
3908    0.553596
5540    0.547355
2165    0.546326
dtype: float64

In [52]:
answer = 4577
print("\n".join([x for x in sent_tokenize(docs[answer])]))

Per preparare la torta libro, iniziate dalla realizzazione del con il metodo diviso a freddo.
Prendete le uova, che dovranno essere a temperatura ambiente, e dividete accuratamente i tuorli dagli albumi.
Ponete quindi gli albumi nella ciotola della planetaria e aggiungete circa 1/3 dello zucchero e non a neve.
Otterrete così un composto spumoso e non a fiocchi; tenetelo da parte e nel frattempo unite la parte restante di zucchero ai tuorli Iniziate a montare anche questi, con uno sbattitore, e una volta ottenuto un composto spumoso, con una buona alveolatura e di colore giallo chiaro, unite i semi della bacca di vaniglia direttamente sul composto,e aiutandovi con una spatola amalgamate delicatamente tutti gli ingredienti Imburrate e infarinate l’apposita teglia, dalle dimensioni 35x28 cm, versatevi il composto Infornate in forno statico preriscaldato a 180°C per 60-70 minuti (160° per circa 50-60 minuti se forno ventilato) .
Una volta sfornato il dolce lasciatelo intiepidire nello stam

### Tokenizzazione per apprendimento

Per un'introduzione sui metodi basati su WOrdPiece si veda [WordPiece Example](./wordpiece.ipynb)

WordPiece è un algoritmo di tokenizzazione basato su sotto-parole, comunemente utilizzato nei modelli basati su trasformatori come BERT. Migliora l'efficienza bilanciando la dimensione del vocabolario e la capacità di rappresentazione. L'obiettivo dell'addestramento è costruire un vocabolario ottimale a partire da dati testuali grezzi, equilibrando la dimensione del vocabolario con l'efficienza delle sotto-parole.

#### Passo 1: Iniziamo con un vocabolario base
Il vocabolario iniziale contiene tutti i singoli caratteri Unicode (A-Z, a-z, 0-9, punteggiatura), oltre a token speciali come `[UNK]`.

#### Passo 2: Calcolare la frequenza di sotto-sequenze di caratteri
Scansioniamo il corpus di addestramento e contiamo la frequenza di ogni coppia di token (inizialmente ogni carattere è considerato un token), contrassegnando con un carattere speciale i token che non sono quelli iniziali.

In [ ]:
import nltk

In [ ]:
corpus = ["mescolo", "mescolare", "mescolavo", "mescoli"]
vocabulary = defaultdict(lambda: 0)
c_prefix = "##"

def update_vocabulary(corpus, vocabulary, c_prefix):
    for text in corpus:
        for i, (a, b) in enumerate(nltk.ngrams(text, n=2)):
            if i == 0:
                vocabulary[(a, f"{c_prefix}{b}")] += 1
            else:
                vocabulary[(f"{c_prefix}{a}", f"{c_prefix}{b}")] += 1

update_vocabulary(corpus=corpus, vocabulary=vocabulary, c_prefix=c_prefix)

pd.Series(vocabulary).sort_values(ascending=False)

#### Passo 3: Combinare le coppie più frequenti e creare nuovi token
Uniamo la coppia di token più frequente e aggiorniamo il vocabolario e il corpus sostituendo questa sequenza con un singolo token.

In [ ]:
pair = list(pd.Series(vocabulary).sort_values(ascending=False).head(1).keys())[0]
new_token = f"{pair[0].replace(c_prefix, '')}{pair[1].replace(c_prefix, '')}"

vocabulary[new_token] = vocabulary[pair]

def wordpiece_split(word, vocabulary):
    tokens = []
    i = 0
    while i < len(word):
        matched = None
        for j in range(len(word), i, -1):
            subword = word[i:j]
            if subword in vocabulary:
                matched = subword
                break
        if matched is None:
            matched = word[i]
        tokens.append(matched)
        i += len(matched)
    return tokens

print(wordpiece_split("mesco", vocabulary=vocabulary))
print(wordpiece_split("ciao", vocabulary=vocabulary))
#print(wordpiece_split("metto", vocabulary=vocabulary))

new_corpus = []
for text in corpus:
    tokens = wordpiece_split(text, vocabulary=vocabulary)
    new_corpus.append(tokens)

corpus = new_corpus

In [ ]:
corpus

#### Passo 4: Ripetere fino a raggiungere la dimensione massima del vocabolario
Da quel momento in poi, tutte le porzioni di stringa che non trovano corrispondenza resteranno inalterate.


In [ ]:
update_vocabulary(corpus=corpus, vocabulary=vocabulary, c_prefix=c_prefix)

In [ ]:
pd.Series(vocabulary).sort_values(ascending=False)

In [ ]:
pair = list(pd.Series(vocabulary).sort_values(ascending=False).head(1).keys())[0]
new_token = f"{pair[0].replace(c_prefix, '')}{pair[1].replace(c_prefix, '')}"

print(new_token)

vocabulary[new_token] = vocabulary[pair]

new_corpus = []
for text in corpus:
    tokens = wordpiece_split("".join(text), vocabulary=vocabulary)
    new_corpus.append(tokens)

corpus = new_corpus

print(corpus)

Nota: questa implementazione ingenua non funziona completamente; per un esempio utilizzabile, puoi controllare qui: [WordPieace](./wordpiece.ipynb)